# 📘 Notebook - Training a Tiny GPT From Scratch

## 🎯 Objective

In this notebook, we'll learn how a GPT model is trained.

By the end of this notebook, you'll understand:

- How training data is prepared
- Context Window
- Input–Target pairs
- Forward Pass
- Loss Function
- Cross Entropy Loss
- Backpropagation
- Gradient Descent
- Optimizer
- Training Loop
- Epochs
- Inference after Training

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
corpus = [
    "I love AI",
    "I love Python",
    "AI loves Python",
    "Python loves AI"
]
for sentence in corpus:
    print(sentence)

I love AI
I love Python
AI loves Python
Python loves AI


In [3]:
vocab = {}
index = 0
for sentence in corpus:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = index
            index += 1
print(vocab)

{'I': 0, 'love': 1, 'AI': 2, 'Python': 3, 'loves': 4}


In [4]:
encoded_corpus = []
for sentence in corpus:
    encoded_sentence = []
    for word in sentence.split():
        encoded_sentence.append(vocab[word])
    encoded_corpus.append(encoded_sentence)
print(encoded_corpus)

[[0, 1, 2], [0, 1, 3], [2, 4, 3], [3, 4, 2]]


In [5]:
data = []
for sentence in encoded_corpus:
    data.extend(sentence)
print(data)

[0, 1, 2, 0, 1, 3, 2, 4, 3, 3, 4, 2]


In [6]:
context_size = 3
print("Context Size:", context_size)

Context Size: 3


In [7]:
inputs = []
targets = []
for i in range(len(data) - context_size):
    inputs.append(data[i:i + context_size])
    targets.append(data[i + context_size])
print("Inputs:")
for inp in inputs:
    print(inp)
print()
print("Targets:")
print(targets)

Inputs:
[0, 1, 2]
[1, 2, 0]
[2, 0, 1]
[0, 1, 3]
[1, 3, 2]
[3, 2, 4]
[2, 4, 3]
[4, 3, 3]
[3, 3, 4]

Targets:
[0, 1, 3, 2, 4, 3, 3, 4, 2]


In [8]:
inputs=torch.tensor(inputs)
targets=torch.tensor(targets)
print("Inputs Shape :", inputs.shape)
print("Targets Shape:", targets.shape)

Inputs Shape : torch.Size([9, 3])
Targets Shape: torch.Size([9])


In [9]:
vocab_size = len(vocab)
embedding_dim = 8
print("Vocabulary Size:", vocab_size)
print("Embedding Dimension:", embedding_dim)

Vocabulary Size: 5
Embedding Dimension: 8


In [10]:
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )
        self.linear = nn.Linear(
            embedding_dim,
            vocab_size
        )
    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        x = self.linear(x)
        return x

In [11]:
model=TinyGPT()
print(model)

TinyGPT(
  (embedding): Embedding(5, 8)
  (linear): Linear(in_features=8, out_features=5, bias=True)
)


In [12]:
logits = model(inputs)
print(logits.shape)

torch.Size([9, 5])


In [13]:
print(logits)

tensor([[-0.2400, -0.6285,  0.5324, -0.3314,  0.8322],
        [-0.2400, -0.6285,  0.5324, -0.3314,  0.8322],
        [-0.2400, -0.6285,  0.5324, -0.3314,  0.8322],
        [-0.0920, -0.7568,  0.7697,  0.0256,  0.7801],
        [-0.2038, -0.4696,  0.6047, -0.1922,  0.2291],
        [-0.0551, -0.0331,  0.4632, -0.1946, -0.4061],
        [-0.0551, -0.0331,  0.4632, -0.1946, -0.4061],
        [ 0.0929, -0.1614,  0.7004,  0.1624, -0.4582],
        [ 0.0929, -0.1614,  0.7004,  0.1624, -0.4582]],
       grad_fn=<AddmmBackward0>)


In [14]:
probabilities = F.softmax(logits, dim=1)
print(probabilities)

tensor([[0.1302, 0.0883, 0.2820, 0.1189, 0.3806],
        [0.1302, 0.0883, 0.2820, 0.1189, 0.3806],
        [0.1302, 0.0883, 0.2820, 0.1189, 0.3806],
        [0.1352, 0.0695, 0.3200, 0.1520, 0.3233],
        [0.1523, 0.1168, 0.3419, 0.1541, 0.2349],
        [0.1896, 0.1938, 0.3183, 0.1649, 0.1335],
        [0.1896, 0.1938, 0.3183, 0.1649, 0.1335],
        [0.1901, 0.1474, 0.3490, 0.2038, 0.1096],
        [0.1901, 0.1474, 0.3490, 0.2038, 0.1096]], grad_fn=<SoftmaxBackward0>)


In [15]:
predictions = torch.argmax(probabilities, dim=1)
print(predictions)

tensor([4, 4, 4, 4, 2, 2, 2, 2, 2])


In [16]:
loss = F.cross_entropy(logits, targets)
print(loss)

tensor(1.7836, grad_fn=<NllLossBackward0>)


In [17]:
optimizer=torch.optim.SGD(
    model.parameters(),
    lr=0.1
)
print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [18]:
optimizer.zero_grad()

In [19]:
loss.backward()

In [20]:
print("Gradients of Embedding Layer:\n")
print(model.embedding.weight.grad)

Gradients of Embedding Layer:

tensor([[ 0.0024,  0.0154, -0.0072,  0.0178,  0.0251,  0.0240, -0.0248,  0.0050],
        [ 0.0040,  0.0120,  0.0002,  0.0050,  0.0175,  0.0164, -0.0193,  0.0035],
        [-0.0118,  0.0273, -0.0073, -0.0136,  0.0519,  0.0109, -0.0303,  0.0142],
        [-0.0147, -0.0105,  0.0017, -0.0257, -0.0197, -0.0392,  0.0344,  0.0037],
        [-0.0160,  0.0041, -0.0066, -0.0157,  0.0112, -0.0186,  0.0089,  0.0079]])


In [21]:
optimizer.step()

In [22]:
epochs=20
for epoch in range(epochs):
    optimizer.zero_grad()
    logits=model(inputs)
    loss=F.cross_entropy(logits,targets)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")
print("Final Loss:", loss.item())

Epoch 1: Loss = 1.7414
Epoch 2: Loss = 1.7030
Epoch 3: Loss = 1.6680
Epoch 4: Loss = 1.6359
Epoch 5: Loss = 1.6066
Epoch 6: Loss = 1.5798
Epoch 7: Loss = 1.5551
Epoch 8: Loss = 1.5324
Epoch 9: Loss = 1.5114
Epoch 10: Loss = 1.4920
Epoch 11: Loss = 1.4739
Epoch 12: Loss = 1.4570
Epoch 13: Loss = 1.4412
Epoch 14: Loss = 1.4263
Epoch 15: Loss = 1.4123
Epoch 16: Loss = 1.3991
Epoch 17: Loss = 1.3866
Epoch 18: Loss = 1.3747
Epoch 19: Loss = 1.3634
Epoch 20: Loss = 1.3526
Final Loss: 1.3525816202163696


In [23]:
prompt = torch.tensor([[0, 1, 2]])
print(prompt)

tensor([[0, 1, 2]])


In [24]:
with torch.no_grad():
    logits = model(prompt)
    prediction = torch.argmax(logits, dim=1)
print("Predicted Token ID:", prediction.item())

Predicted Token ID: 3


In [25]:
reverse_vocab = {}
for word, idx in vocab.items():
    reverse_vocab[idx] = word
predicted_word = reverse_vocab[prediction.item()]
print("Predicted Word:", predicted_word)

Predicted Word: Python


In [26]:
sentence = ["I", "love", "AI"]
sentence.append(predicted_word)
print("Generated Sentence:")
print(" ".join(sentence))

Generated Sentence:
I love AI Python
